
# Storied SDP RAG: LlamaIndex + FAISS + RAGAS + TruLens

End-to-end workflow to ingest existing `.sdp` (JSON) reports, index them locally, query with a local LLM, and optionally evaluate with RAGAS + TruLens for research-grade transparency.



## 0. Install dependencies (run once per environment)
- Uses local models via Ollama by default. Swap `Ollama` for another LLM in the config cell if needed.
- The list is intentionally explicit so versions are easy to control for research runs.


In [2]:

%pip install -q     "llama-index>=0.10.30"     "llama-index-vector-stores-faiss>=0.1.1"     "llama-index-embeddings-huggingface>=0.1.1"     "faiss-cpu>=1.7.4"     "trulens-eval>=0.21.1"     "ragas>=0.1.7"     "datasets>=2.16"     "langchain-community>=0.0.21"     "sentence-transformers>=2.2"     "ollama>=0.1.4"


Note: you may need to restart the kernel to use updated packages.



## 1. Configuration
- Point `DATA_DIR` to your `.sdp` JSON folder.
- Adjust `LLM_MODEL` to the local model you pulled with Ollama (e.g., `llama2`, `mistral`, `mixtral`).
- `EMBED_MODEL` uses a light, license-friendly encoder; swap if you prefer a domain model.


In [3]:

from pathlib import Path
import os

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.file.base import SimpleDirectoryReader
from llama_index.node_parser import SentenceSplitter
from llama_index import VectorStoreIndex, ServiceContext, StorageContext
from llama_index.vector_stores.faiss import FaissVectorStore

# Paths and models
DATA_DIR = Path("storied/data")
INDEX_DIR = Path("storied/index/sdp_faiss")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

LLM_MODEL = os.environ.get("SDP_LLM_MODEL", "llama2")  # pulled via `ollama pull llama2`
EMBED_MODEL = os.environ.get("SDP_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

docs = SimpleDirectoryReader(
    DATA_DIR,
    recursive=True,
    required_exts=[".sdp"],
    filename_as_id=True,
).load_data()
print(f"Loaded {len(docs)} documents from {DATA_DIR}")

llm = Ollama(model=LLM_MODEL)
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(docs)

service_context = ServiceContext.from_defaults(
    llm=llm,
    embed_model=embed_model,
)

vector_store = FaissVectorStore(dimension=embed_model.embedding_dim)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    service_context=service_context,
)
index.storage_context.persist(persist_dir=str(INDEX_DIR))
print(f"Index persisted to {INDEX_DIR}")


ModuleNotFoundError: No module named 'llama_index.llms.ollama'


## 2. Query the indexed SDP reports
Use LlamaIndex `QueryEngine` to retrieve relevant chunks and synthesize an answer with the local LLM.


In [ ]:

query_engine = index.as_query_engine(similarity_top_k=4, response_mode="compact")

sample_question = "Summarize the key findings and recommendations in the SDP report."
response = query_engine.query(sample_question)
print(response)

# Inspect retrieved chunks for transparency
for i, node in enumerate(response.source_nodes, start=1):
    print(f"
[Context {i}] {node.score:.2f}
{node.node.get_content()[:400]}")



## 3. RAGAS evaluation (optional, research-grade)
- Generates a small eval set of questions and runs RAGAS metrics (`answer_relevancy`, `faithfulness`, `context_recall`).
- Replace `GROUND_TRUTHS` with SME-curated answers for more meaningful scores.


In [ ]:

from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_recall
from datasets import Dataset

# Lightweight eval prompts (add more or load from file)
eval_questions = [
    "What is the primary objective of this report?",
    "List any KPIs or metrics highlighted.",
    "What actions or recommendations are proposed?",
]

# TODO: replace with SME/ground-truth answers per question for stronger evaluation
GROUND_TRUTHS = {
    "What is the primary objective of this report?": "<fill from report>",
    "List any KPIs or metrics highlighted.": "<fill from report>",
    "What actions or recommendations are proposed?": "<fill from report>",
}

records = []
for q in eval_questions:
    res = query_engine.query(q)
    records.append(
        {
            "question": q,
            "answer": str(res),
            "contexts": [n.node.get_content() for n in res.source_nodes],
            "ground_truth": GROUND_TRUTHS.get(q, ""),
        }
    )

dataset = Dataset.from_list(records)
ragas_result = evaluate(
    dataset,
    metrics=[answer_relevancy, faithfulness, context_recall],
)
print(ragas_result)



## 4. TruLens tracing (optional)
- Wrap the query engine with TruLens to log traces/feedback for observability.
- Add feedback functions (groundedness, toxicity, etc.) as needed.


In [ ]:

from trulens_eval import Tru, TruLlama

tru = Tru()  # defaults to sqlite local DB
tru.reset_database()

tru_query_engine = TruLlama(
    query_engine,
    app_id="sdp-rag",
)

with tru_query_engine as recording:
    traced_response = query_engine.query(
        "Provide a concise executive summary of the SDP report in 3 bullet points."
    )

print(traced_response)
records, feedback = tru.get_records_and_feedback(app_ids=["sdp-rag"])
print(f"Recorded {len(records)} traces. View dashboard with tru.get_leaderboard().")



## 5. Next steps
- Swap FAISS with Chroma or a remote store if you outgrow local storage.
- Tune chunking, embeddings, and prompt templates for domain accuracy.
- Expand the evaluation set and ground truths to make RAGAS scores meaningful.
- Add TruLens feedback functions (e.g., groundedness, toxicity) for deeper traces.
